# 12 · Continuous batching and chunked prefill
Level L3 · Part 2 · core · ~25 min · builds on 11

## Goal
By the end of this lesson you can explain iteration-level scheduling, read the scheduler defaults of the pinned vLLM image, and measure how chunked prefill trades TTFT against the stalls that other requests feel (ITL).

## Why it matters
- **Without it (static batching):** the engine runs a batch until its longest answer ends and admits nothing new meanwhile. In the first eight requests of the trace, answers need 24 to 113 tokens, so the short ones leave slots idle.
- **Without chunked prefill:** one 1,600-token prompt takes a whole step by itself. Every request decoding in that step waits, so their streamed tokens stall.
- **With them:** a request joins a free slot on the next step, and a long prompt is split into chunks. The cost: a slower first token for that prompt.
- **When you don't need it:** offline batches of equal-length requests.

## The idea
Think of a bus that leaves on a timetable and takes whoever is at the stop. Static batching is a bus that waits for a full load, drives the whole route, and only then picks up the next group. Continuous batching is a bus that stops at every stop: a passenger who gets off frees a seat and the next one in the queue gets on. vLLM's V1 scheduler works this way: each step it gives every running request the tokens it needs, up to a token budget (the most tokens one step may process: `max_num_batched_tokens`). Chunked prefill lets a long prompt use only the part of the budget that is left, so it arrives over several steps.

## Picture
```mermaid
flowchart TB
  S["one engine step, token budget B"] --> D["1. each running request past its prompt: 1 decode token"]
  D --> P["2. prefill takes the rest: a whole prompt, or a chunk when chunked prefill is on"]
  P --> F["3. finished requests leave; waiting requests join on the next step"]
  F --> S
```

## Step by step
### Step 1: the scheduler defaults in the pinned image
The vLLM CPU image (`vllm/vllm-openai-cpu:v0.32.0`, the same tag as lessons 08 and 11) contains the scheduler. The probe below imports its `SchedulerConfig` and prints the defaults, then checks that the V1 scheduler is in the image. It needs no model. If Docker is not enabled here, the next cell prints the recorded output from the same run.

In [1]:
LIBRARY_PROBE = """
import inspect
from vllm.config import SchedulerConfig
from vllm.v1.core.sched.scheduler import Scheduler
c = SchedulerConfig(max_model_len=4096, is_encoder_decoder=False)
print('max_num_seqs:', c.max_num_seqs)
print('max_num_batched_tokens:', c.max_num_batched_tokens)
print('enable_chunked_prefill:', c.enable_chunked_prefill)
print('scheduler class:', Scheduler.__module__ + '.' + Scheduler.__name__)
print('scheduler comment:', [l.strip() for l in inspect.getsource(Scheduler.schedule).splitlines() if 'no "decoding phase"' in l][0])
"""

**Run the probe in the pinned image.** Docker starts the image's Python with `-I`, so the run ignores the working folder. This cell is skipped unless Docker is enabled.

In [2]:
import subprocess
from pathlib import Path

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"           # the pinned CPU image (lessons 08 and 11 use the same tag)
Path("../data/12").mkdir(parents=True, exist_ok=True)
run = subprocess.run(["docker", "run", "--rm", "--entrypoint", "python3", IMAGE, "-I", "-c", LIBRARY_PROBE],
                     capture_output=True, text=True, timeout=900)
if run.returncode == 0:
    Path("../data/12/library_probe.txt").write_text(run.stdout)
    print(run.stdout)
else:
    print("docker run failed:", run.stderr[-300:])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**The same output without Docker.** The cell reads the live output if the cell above ran, and otherwise the recorded copy in `data/recorded/12.json`.

In [3]:
import json
from pathlib import Path
live = Path("../data/12/library_probe.txt")
if live.exists():
    print("live output from the pinned image:")
    print(live.read_text())
else:
    recorded = json.loads(Path("../data/recorded/12.json").read_text())
    print("recorded output (from a real run of the same image, " + recorded["image"] + "):")
    print(recorded["stdout"])

live output from the pinned image:
max_num_seqs: 128
max_num_batched_tokens: 2048
enable_chunked_prefill: True
scheduler class: vllm.v1.core.sched.scheduler.Scheduler
scheduler comment: # There's no "decoding phase" nor "prefill phase" in the scheduler.



**What you should see:** `max_num_seqs: 128`, `max_num_batched_tokens: 2048`, and `enable_chunked_prefill: True`. These are the class defaults. The engine's startup log prints the effective values for your model and GPU, so check there before you tune. The scheduler comment says the step has no prefill or decode phase; each request only has to catch up to the tokens it owns.

## A small simulator
This is a cost-model teaching device, not a library: vLLM's scheduler lives inside the engine and has no API to call. The code below repeats its rule in about 40 lines. Each step, decoders get one token each, and prefill takes the rest of the token budget. With chunking off, a prompt needs its whole remaining length to fit or it waits. The step time is an illustrative cost model (a fixed overhead plus work per token), not a measurement.

In [4]:
import json
from pathlib import Path
trace = [json.loads(l) for l in Path("../data/07/support_trace.jsonl").read_text().splitlines()]  # lesson 07 shows its first rows
BASE_MS, PER_TOKEN_MS = 8.0, 0.04   # illustrative step cost, not measured

def schedule(running, mode, budget):
    """Tokens per request in one step: decoders first (one each), then prefill takes what is left."""
    left, work = budget, {}
    for r in running:                                   # decoders
        if r["done"] >= r["prompt"]: work[r["id"]] = 1; left -= 1
    for r in running:                                   # prefill
        need = r["prompt"] - r["done"]
        take = min(need, left) if mode == "chunked" else need
        if 0 < take <= left: work[r["id"]] = take; left -= take
    return work

`simulate` runs the loop: admit, schedule, advance the clock, apply the step to each request. The three modes differ only in the admission rule and in whether a prompt may be split.

In [5]:
def simulate(mode, budget=2048, max_seqs=128, gap_ms=20.0):
    """static: admit only into an empty batch; continuous: whole prompts only; chunked: split prompts to fit."""
    if mode != "chunked" and max(r["prompt_tokens"] for r in trace) > budget:
        raise ValueError(f"a {max(r['prompt_tokens'] for r in trace)}-token prompt cannot fit a {budget}-token step without chunked prefill")
    reqs = [dict(id=i, arrive=i * gap_ms, prompt=r["prompt_tokens"], out=r["output_tokens"],
                 done=0, made=0, last=None, gaps=[]) for i, r in enumerate(trace)]
    waiting, running, finished, clock = list(reqs), [], [], 0.0
    while len(finished) < len(reqs):
        if mode != "static" or not running:            # admit arrived requests into free slots
            for r in [r for r in waiting if r["arrive"] <= clock][:max_seqs - len(running)]:
                running.append(r); waiting.remove(r)
        if not running:
            clock = waiting[0]["arrive"]; continue
        work = schedule(running, mode, budget)
        clock += BASE_MS + PER_TOKEN_MS * sum(work.values())
        for r in [r for r in running if r["id"] in work]:   # apply the step to each request that got tokens
            r["done"] += work[r["id"]]
            if r["done"] < r["prompt"]: continue        # still in prefill: no token yet
            r["made"] += 1                              # the first token, or the next one
            if r["made"] > 1: r["gaps"].append(clock - r["last"])
            r.setdefault("first", clock); r["last"] = clock
            if r["made"] >= r["out"]:
                r["end"] = clock; running.remove(r); finished.append(r)
    return reqs

**Reading `simulate`:** the `static` mode admits new requests only when the batch is empty. Nobody new joins until every running request finishes, so short requests leave empty slots. That wait is why its first tokens are slow in the table below.

### Step 2: measure the three modes on the same 30 requests
The trace is the made-up support-desk trace from lesson 07: 30 requests, prompts of about 1,500 to 1,620 tokens (the shared policy prompt plus a question), 24 to 199 output tokens. Requests arrive every 20 ms. The table reports TTFT (time to first token), the 95th percentile of ITL, and tokens per second. ITL is the gap between two streamed tokens. TPOT, from lessons 06 and 07, is its average, so the ITL tail shows the stalls that TPOT hides.

In [6]:
def summary(reqs):
    ttft = sorted(r["first"] - r["arrive"] for r in reqs)
    itl = sorted(g for r in reqs for g in r["gaps"])
    span = max(r["end"] for r in reqs) - min(r["arrive"] for r in reqs)
    pick = lambda xs, q: xs[min(len(xs) - 1, int(q * len(xs)))]
    return {"TTFT p50 ms": pick(ttft, 0.5), "TTFT p95 ms": pick(ttft, 0.95),
            "ITL p95 ms": pick(itl, 0.95), "tokens/s": 1000 * sum(r["out"] for r in reqs) / span}

print("longest prompt in the trace:", max(r["prompt_tokens"] for r in trace), "tokens")
print(f"{'mode':<12}{'TTFT p50':>10}{'TTFT p95':>10}{'ITL p95':>10}{'tokens/s':>10}")
for mode in ("static", "continuous", "chunked"):
    s = summary(simulate(mode))
    print(f"{mode:<12}{s['TTFT p50 ms']:>10.0f}{s['TTFT p95 ms']:>10.0f}{s['ITL p95 ms']:>10.0f}{s['tokens/s']:>10.0f}")

longest prompt in the trace: 1617 tokens
mode          TTFT p50  TTFT p95   ITL p95  tokens/s
static            1870      2521        71       441
continuous         832      1489        71       563
chunked            849      1489        90       568


**What you should see:** static batching has the slowest first tokens (p50 about 1,870 ms) and the lowest tokens per second (441), because slots stay empty while the batch drains. Continuous batching fixes both (832 ms, 563 tokens/s) and keeps the same ITL tail, 71 ms, which the long prompts cause. Chunked prefill at the default budget of 2,048 does not lower that tail here (90 ms): every step is filled to the budget, and the budget is larger than the prompts. The milliseconds come from the cost model above, so trust the ranking, not the numbers.

**Worked example: one step at budget 2,048.** Suppose 100 requests are decoding and two 1,617-token prompts are waiting. The decoders take 100 tokens, so 1,948 are left. Continuous batching admits the first prompt whole (1,617 fits), then holds the second, because only 331 tokens are left. That step carries 1,717 tokens and costs 8 + 0.04 × 1,717 ≈ 77 ms. Chunked prefill gives the second prompt those 331 tokens, so the step carries 2,048 tokens and costs 8 + 0.04 × 2,048 ≈ 90 ms. Chunking packs more prefill into each step, so the prompts finish sooner, and the fuller step is what the decoders pay in ITL.

### Step 3: the budget decides the trade-off
Chunked prefill with a smaller budget leaves more room for decoders in each step, so the ITL tail gets shorter. The same long prompt now needs several steps, so its own TTFT grows. Run the chunked mode with two budgets.

In [7]:
print(f"{'budget':<12}{'TTFT p50':>10}{'TTFT p95':>10}{'ITL p95':>10}{'tokens/s':>10}")
for budget in (2048, 512):
    s = summary(simulate("chunked", budget=budget))
    print(f"{budget:<12}{s['TTFT p50 ms']:>10.0f}{s['TTFT p95 ms']:>10.0f}{s['ITL p95 ms']:>10.0f}{s['tokens/s']:>10.0f}")

budget        TTFT p50  TTFT p95   ITL p95  tokens/s
2048               849      1489        90       568
512               1124      2003        28       516


**What you should see:** with 512 tokens per step the ITL p95 falls from 90 ms to 28 ms, and the TTFT p50 rises from 849 ms to 1,124 ms. That is the TTFT versus TPOT trade-off of chunked prefill: when the budget is smaller than the prompts, no step carries a whole long prompt, so decoders keep their rhythm, and the long prompt waits longer for its first token. vLLM's tuning guide says the same: a smaller `max_num_batched_tokens` favours inter-token latency, a larger one favours TTFT.

## At work
- **Databricks** reports that continuous batching gave 10 to 20 times the throughput of dynamic batching, and for MPT-7B at batch 64 about 14 times the throughput for about 4 times the latency. Measure both numbers on your own load. → lesson 27 (load tests).
- **Cloudflare** reports that when prefill and decode were separated, p90 inter-token latency fell from about 100 ms with high variance to 20 to 30 ms. Separating prefill from decode is the heavier fix for the same interference. → lesson 34 (disaggregated prefill).
- **Head-of-line blocking** is the usual production symptom: one 100,000-token prompt stalls every decoder, so the median looks fine while the TPOT p99 spikes. Watch the ITL tail, not only the average. → lesson 33 (reliability drill).

Sources: Databricks, "LLM Inference Performance Engineering: Best Practices" (databricks.com/blog); Cloudflare, "High-performance LLMs" (2026-05, blog.cloudflare.com/high-performance-llms/); vLLM optimization docs (docs.vllm.ai).

## Common mistakes
**1. Sending a prompt longer than the budget to an engine that cannot chunk.** Without chunked prefill, a prompt must fit one step. The simulator refuses it, and an engine without chunked prefill has a similar limit.

In [8]:
simulate("continuous", budget=1000)     # ValueError: a 1617-token prompt cannot fit a 1000-token step without chunked prefill

ValueError: a 1617-token prompt cannot fit a 1000-token step without chunked prefill

**2. Reading `max_num_seqs` as the throughput knob.** It caps how many requests run at once, but the token budget decides how much work one step can carry. Raising the cap does not help once the budget is full: with a 512-token budget, 128 decoders take 128 tokens and prefill gets the other 384, so extra requests wait in the same step. Keep `max_seqs` below the budget: every running decoder takes a token from each step, so a cap above the budget can leave prefill nothing. Tune both together and watch the step time and the ITL tail.

## Try it
Predict first: for `chunked` with a budget of 1024 and then 256, will the ITL p95 and the TTFT p50 be higher or lower than at 512? Write down the direction for each, then run the cell and compare the printed numbers with your prediction.

In [9]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [10]:
for budget in (1024, 256):
    s = summary(simulate("chunked", budget=budget))
    print(f"budget {budget}: TTFT p50 {s['TTFT p50 ms']:.0f} ms, ITL p95 {s['ITL p95 ms']:.0f} ms")
# Printed: budget 1024: TTFT p50 924 ms, ITL p95 49 ms; budget 256: TTFT p50 1524 ms, ITL p95 18 ms.
# Against 512 (TTFT p50 1124 ms, ITL p95 28 ms): a bigger budget gives a higher ITL tail and a lower TTFT;
# a smaller one gives a lower ITL tail and a higher TTFT. Decoders get a token in every step either way,
# but each long prompt needs more steps to finish its prefill when the budget is small.

budget 1024: TTFT p50 924 ms, ITL p95 49 ms
budget 256: TTFT p50 1524 ms, ITL p95 18 ms


## Recap
- Continuous batching admits a request on any step and frees its slot when it finishes, so slots do not wait for the slowest request in the batch.
- Each step has a token budget. Decoders get one token each first, then prefill fills the rest.
- Chunked prefill splits a long prompt over several steps. With a budget smaller than the prompts it lowers the ITL tail and raises the TTFT of the long prompt.
- Smaller budgets favour inter-token latency, larger budgets favour TTFT. Tune them against your SLO, not against the default.

**Interview:** "What is continuous batching, and what does chunked prefill trade off?" Strong answer: the scheduler works per iteration, not per batch; each step has a token budget that decoders fill first; chunked prefill caps how much one prompt can stall the decoders, at the cost of more steps, and so a later first token, for the long prompt.